# Wireless Coverage — Part 3: H3 Gridding of DEM / DSM / CHM Surfaces

Prepare the LiDAR-derived elevation surfaces from Part 2 for global-grid analysis by
H3-indexing each surface. DEM and DSM elevation bands are extracted as contiguous isoband
polygons (`rst_isoband`) and indexed to H3 cells with the Databricks product
`h3_try_coverash3`. The Canopy Height Model is aggregated to the maximum canopy height per
H3 cell (`gbx_rst_h3_rastertogridmax`). Gaps over land are filled with `h3_cellfill`
(k=1, IDW). The resulting H3 tables (`wc_h3_{dem,dsm,chm}_res10`) are prepared for
downstream coverage analysis — antenna siting and line-of-sight in Part 4.

Shared configuration (imports, params, helpers) is loaded via `%run ./config_nb` after
installation. Notebook-specific overrides follow the `%run` cell.

> **Runtime.** Runs on **Databricks Serverless environment 6** — the lightweight tier (pure
> Python/PySpark, no JAR or GDAL init script). Set `DEMO=True` (default) for a Golden Gate
> Park sub-AOI (~25 tiles) to iterate quickly; `DEMO=False` for full San Francisco.
> **H3 resolution:** 10 (≈ 65 m edge length — series default from `config_nb`).

---
**Last Update:** 2026-10-03


In [0]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall \
  "geobrix[light_env6,vizx,overture] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet \
  "geobrix[light_env6,vizx,overture] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"


In [0]:
%restart_python


In [0]:
%run ./config_nb

In [0]:
# ── nb3-specific overrides (applied after %run ./config_nb) ─────────────────
# H3_RESOLUTION = 10 comes from config_nb (series default).
SRC_PREFIX   = "wc_surface"   # source tables from nb2: wc_surface_{dsm,dtm,chm}
OUT_PREFIX   = "wc_h3"        # H3 output tables:  wc_h3_{dem,dsm,chm}_res{H3_RESOLUTION}
LAND_DIR     = "/Volumes/geospatial_docs/geobrix/sample-data/geobrix-examples/sf/overture-water-bay"
BREAKS_M     = [float(b) for b in range(0, 301, 30)]   # 10 elevation bands × 30 m
DEMO         = True
RENDER_H3    = True
INTERACTIVE  = False   # True → live deck.gl; False → static (GitHub-safe)

In [0]:
dsm = spark.table(_tbl(SRC_PREFIX, "dsm"))
dtm = spark.table(_tbl(SRC_PREFIX, "dtm"))
chm = spark.table(_tbl(SRC_PREFIX, "chm"))
print(f"Source surfaces: DSM {dsm.count()} | DTM {dtm.count()} | CHM {chm.count()} tiles")

# Confirm CRS (all should be EPSG:3857 — requires rst_transform before H3 indexing).
sample_srid = dsm.select(F.expr("gbx_rst_srid(dsm)").alias("srid")).first()["srid"]
assert sample_srid == 3857, f"Expected EPSG:3857, got {sample_srid}"
print(f"Surface CRS confirmed: EPSG:{sample_srid}")


In [0]:
if RENDER_H3:
    laz_files = sorted(glob.glob(f"{LAZ_DIR}/**/*.laz", recursive=True))
    if laz_files:
        if INTERACTIVE:
            # Live deck.gl orbit view — Databricks interactive sessions only.
            plot_point_cloud(
                laz_files[0], color="rgb", max_points=150_000,
                title="SF LiDAR — raw returns (one EPT node, RGB-colored)",
            )
        else:
            # Static Axes3D scatter — image/png, GitHub-safe.
            plot_point_cloud(
                laz_files[0], color="rgb", max_points=150_000,
                max_embed_mb=0,
                title="SF LiDAR — raw returns (one EPT node, RGB-colored)",
            )
    else:
        print(f"No .laz staged under {LAZ_DIR}; point cloud preview skipped.")


In [0]:

if DEMO:
    _gx, _gy = lonlat_to_3857(-122.486, 37.769)   # Golden Gate Park centroid
    _tx, _ty = int((_gx - X0) / TILE_M), int((_gy - Y0) / TILE_M)
    DEMO_FILTER = (
        F.col("tx").between(_tx - 2, _tx + 2) &
        F.col("ty").between(_ty - 2, _ty + 2)
    )
    dtm_in = dtm.where(DEMO_FILTER)
    dsm_in = dsm.where(DEMO_FILTER)
    chm_in = chm.where(DEMO_FILTER)
    print(f"DEMO mode: {dtm_in.count()} DTM | {dsm_in.count()} DSM | {chm_in.count()} CHM tiles "
          f"in ±2 km window around Golden Gate Park (tx={_tx}, ty={_ty})")
else:
    dtm_in, dsm_in, chm_in = dtm, dsm, chm


## Part 3 — T2: DEM / DSM surfaces → elevation isobands → H3 cells

Turns the EPSG:3857 DTM and DSM surfaces from Part 2 into elevation-band H3 cell tables
(`wc_h3_dem`, `wc_h3_dsm`). Key steps:

1. **Land mask** (Overture water features → SF land polygon) — clips Bay and Pacific
   pixels to NoData before isoband extraction.
2. **CRS transform**: `rst_transform(tile, 4326)` — `h3_try_coverash3` requires lon/lat WKB.
3. **`rst_clip`** — apply land mask to each reprojected tile.
4. **`rst_isoband`** — extract contiguous elevation-band polygons.
5. **`h3_try_coverash3`** — index each band polygon to H3 cells (no driver-side indexing).
6. **Persist** both tables; print counts and band distribution.
7. **Cumulative tiers** and a static band map.

In [0]:
# --- Land mask: Overture base/water → SF land polygon ---
# Subtract Bay and Pacific water features from the AOI box to get a land multipolygon.
# Used as the rst_clip cutline to set water/ocean pixels to NoData before isoband.
ov = OvertureClient()
if not glob.glob(f"{LAND_DIR}/**/*.parquet", recursive=True):
    assets = ov.discover(SF_AOI, themes=["base"]).filter(F.col("type") == "water")
    ov.download(assets, LAND_DIR, bbox=SF_AOI)
water = ov.read(LAND_DIR, theme="base", type="water", bbox=SF_AOI)
_wg = [_wkb.loads(bytes(r["geometry"])) for r in water.select("geometry").collect()]
W, S, E, N = SF_AOI
land_geom = _box(W, S, E, N).difference(unary_union(_wg)) if _wg else _box(W, S, E, N)
land_wkb  = land_geom.wkb     # EPSG:4326 land multipolygon (WKB) — cutline for rst_clip
print(f"Land mask: subtracted {len(_wg)} water feature(s); "
      f"land area \u2248 {land_geom.area / _box(W,S,E,N).area:.2f} of AOI")


In [0]:
# --- Elevation-band H3 cells: EPSG:3857 surface tiles → H3 cells (EPSG:4326) ---
# breaks_arr is broadcast to every executor so each call to rst_isoband uses the
# same set of elevation thresholds.
breaks_arr = F.array(*[F.lit(b) for b in BREAKS_M])
# _surface_to_h3_cells(surface_df, tile_col, breaks_array, h3_res, land_wkb=None)
# → defined in config_nb; called with (surface_df, tile_col, breaks_arr, H3_RESOLUTION, land_wkb)


In [0]:
# --- Isoband validation: assert >=3 non-empty elevation bands on a sample ---
# Runs before the full DEM/DSM passes to catch a misconfigured BREAKS_M or a
# mismatched CRS (e.g. tiles still in EPSG:3857 — all elevations would be outside
# the 0-300 m range and yield 0 bands). Sample 5 tiles so the elevation spread
# is enough to span at least 3 bands even in the flat DEMO AOI.
sample_tile = dtm_in.limit(5)
sample_cells = _surface_to_h3_cells(sample_tile, "dtm", breaks_arr, H3_RESOLUTION, land_wkb)
band_counts = sample_cells.groupBy("band_level", "elev_lo", "elev_hi").count().orderBy("band_level")
band_counts.show()
n_nonempty = band_counts.count()
assert n_nonempty >= 3, (
    f"Expected ≥3 non-empty elevation bands, got {n_nonempty}. "
    "Check BREAKS_M range vs actual DTM elevation distribution."
)
print(f"Band validation: {n_nonempty} non-empty elevation bands — OK")

In [0]:
# --- Produce and persist wc_h3_dem (bare earth DTM elevation bands) ---
dem_cells = _surface_to_h3_cells(dtm_in, "dtm", breaks_arr, H3_RESOLUTION, land_wkb)   # DEM = bare earth
dem_h3 = _persist(dem_cells, "dem", res=H3_RESOLUTION)   # → geospatial_docs.geobrix.wc_h3_dem_res10
print(f"wc_h3_dem: {dem_h3.count()} rows (H3 res {H3_RESOLUTION})")
dem_h3.groupBy("band_level", "elev_lo", "elev_hi").count().orderBy("band_level").show()


In [0]:
# --- Produce and persist wc_h3_dsm (surface incl. buildings/canopy) ---
dsm_cells = _surface_to_h3_cells(dsm_in, "dsm", breaks_arr, H3_RESOLUTION, land_wkb)   # DSM = surface incl. buildings/canopy
dsm_h3 = _persist(dsm_cells, "dsm", res=H3_RESOLUTION)   # → geospatial_docs.geobrix.wc_h3_dsm_res10
print(f"wc_h3_dsm: {dsm_h3.count()} rows (H3 res {H3_RESOLUTION})")


In [0]:
# --- DEM cumulative tiers: tier T = 'elevation >= BREAKS_M[T]' ---
# A cell in interval band i belongs to tiers 0..i; coverage depth climbs with
# terrain height — the telco nested-coverage model applied to elevation.
dem_tiers = (
    dem_h3
    .select("cellid", "res",
            F.explode(F.sequence(F.lit(0), F.col("band_level"))).alias("tier"))
    .distinct()
)
dem_tiers.groupBy("tier").count().orderBy("tier").show()


In [0]:
if RENDER_H3:
    # --- Visualize DEM H3 cells: dissolve by band → <=10 footprints, fast render ---
    dem_gdf = cells_as_gdf(
        dem_h3, "cellid",
        extra_cols=["band_level"],
        dissolve_by="band_level",
        dissolve_engine="product",
    )
    plot_static(
        dem_gdf, column="band_level", cmap="terrain",
        title=f"DEM (bare earth) — elevation bands, H3 res {H3_RESOLUTION}",
    )


## CHM → H3 max canopy height

The `gbx_rst_h3_rastertogridmax` UDTF aggregates all raster pixels whose centroid falls in
each H3 cell and returns the maximum value per cell. Applied to the CHM (canopy height in
metres), it gives the tallest vegetation or structure per H3 hexagon — the surface height
signal that nb4 will use for line-of-sight analysis.

**Scale note:** `DEMO=True` (default) runs on ~25 tiles (Golden Gate Park ±2 km); the full
San Francisco AOI is 336 CHM tiles (~336 M pixels at 1 m resolution). At res 9 over full SF,
expect 5–15 min on Serverless — profile with `DEMO=True` first and set `DEMO=False` when
the output is validated.


In [0]:
# rst_h3_rastertogridmax (and all H3 UDTFs) require lon/lat EPSG:4326 tiles.
chm_4326 = chm_in.select(
    "tx", "ty",
    rx.rst_transform("chm", F.lit(4326)).alias("chm_4326"),
)
# Register as a temp view for the SQL LATERAL call below.
# Serverless: repartition(N, column) for fan-out parallelism (sparkContext unavailable).
chm_4326.repartition(64, "tx") \
        .createOrReplaceTempView("chm_4326_tiles")
print(f"CHM tiles prepared for H3 rastertogridmax: {chm_4326.count()} tile(s) at EPSG:4326")


In [0]:
# SQL LATERAL UDTF (ANSI form): invoke via spark.sql — the Python wrapper raises NotImplementedError.
# VERIFIED output schema (T0, live cluster): (band INT, cellID LONG, measure DOUBLE); band=1 is the
# single CHM band. Do NOT use `LATERAL VIEW OUTER ... AS a,b,c` — positional renaming mislabels the
# columns (band->a, cellID->b, measure->c). Use ANSI LATERAL with the real column names:
chm_cells_raw = spark.sql(f"""
    SELECT t.cellID AS cellid, t.measure AS chm_z
    FROM   chm_4326_tiles,
           LATERAL gbx_rst_h3_rastertogridmax(chm_4326, {H3_RESOLUTION}) t
    WHERE  t.band = 1
""")
# Multiple tiles may map the same cellid at tile boundaries → take max across tiles.
chm_cells = (
    chm_cells_raw
    .groupBy("cellid").agg(F.max("chm_z").alias("max_chm_z"))
    .withColumn("res", F.lit(H3_RESOLUTION))
    .select("cellid", "res", "max_chm_z")
)
print(f"CHM H3 cells (before persist): {chm_cells.count()}")
chm_cells.selectExpr("min(max_chm_z)", "max(max_chm_z)", "avg(max_chm_z)").show()


In [0]:
chm_h3 = _persist(chm_cells, "chm", res=H3_RESOLUTION)   # → geospatial_docs.geobrix.wc_h3_chm_res10
_cmin = chm_h3.selectExpr("min(max_chm_z)").first()[0]
_cmax = chm_h3.selectExpr("max(max_chm_z)").first()[0]
_cmin = _cmin if _cmin is not None else float("nan")
_cmax = _cmax if _cmax is not None else float("nan")
print(f"wc_h3_chm: {chm_h3.count()} cells | max_chm_z: [{_cmin:.1f} m, {_cmax:.1f} m]")

## Fill interior gaps with `h3_cellfill`

**Data quality note.** The upstream `wc_surface_chm` table (nb2: `rst_chm` = DSM − DTM,
max all-return height above bare earth) carries LiDAR artifacts in a small fraction of
cells: multi-path reflections, flying objects, or localised DTM underestimates can
produce `max_chm_z` values far above any real structure. A configurable `MAX_CHM_M`
ceiling drops these outliers before gap-filling, so the fill propagates from valid
neighbours rather than from artifact values. The default (350 m) accommodates the
tallest structures in San Francisco (Salesforce Tower ≈ 326 m) while discarding values
that cannot represent real objects. Lower it for vegetation-only AOIs (e.g. 80 m for
a park canopy).

The CHM H3 surface has gaps at NoData areas (edges of the LiDAR survey, water bodies
already removed by land clip). GeoBrix `h3_cellfill` fills missing cells within k=1
rings of a valid neighbour using inverse-distance weighting (IDW, power=2). This keeps
fills strictly local — a gap with no filled neighbour within one ring stays empty.

The fill is applied to the unique `(cellid, max_chm_z)` surface — one value per cell,
so the fill never collapses multiple assignments. After fill, the land mask is
re-applied to prevent any ocean-adjacent cells from being populated.

Shown as an option (identical treatment to the H3 Rasterize extra notebook); the
pipeline works without it.

In [0]:
# --- Configurable artifact ceiling: drop non-physical CHM outliers before gap-fill ---
# CHM = DSM − DTM includes vegetation AND building heights (rst_chm in nb2 bins
# all returns at max z, so towers read as CHM). SF's tallest structure (Salesforce
# Tower) is ~326 m, so 350 m retains all real objects while discarding LiDAR
# artifacts (multi-path reflections, flying objects, DTM underestimates).
# Lower this for vegetation-only AOIs (e.g. 80 m for a park canopy).
MAX_CHM_M = 350.0

chm_h3_clean = chm_h3.where(F.col("max_chm_z") <= MAX_CHM_M)
total_cells   = chm_h3.count()
clean_cells   = chm_h3_clean.count()
dropped       = total_cells - clean_cells
post_max_raw  = chm_h3_clean.selectExpr("max(max_chm_z)").first()[0]
post_max_raw  = post_max_raw if post_max_raw is not None else float("nan")
print(f"MAX_CHM_M = {MAX_CHM_M} m: {dropped} outlier cell(s) dropped "
      f"({100 * dropped / max(total_cells, 1):.0f}% of {total_cells}); "
      f"post-clamp max = {post_max_raw:.1f} m")

In [0]:
# Build the set of valid land H3 cells once (reused for post-fill masking).
# h3_try_coverash3 returns an array; explode + isNotNull filters holes at the
# AOI boundary.  This is the same land polygon (land_wkb) used for rst_clip in T2.
land_cells = (
    spark.createDataFrame([(1,)], "g int")
    .select(F.explode(DBF.h3_try_coverash3(F.lit(land_wkb), F.lit(H3_RESOLUTION))).alias("cellid"))
    .where(F.col("cellid").isNotNull())
    .select("cellid")
)
print(f"Land H3 cells at res {H3_RESOLUTION}: {land_cells.count()} cells")

In [0]:
chm_filled = _cellfill_depth(chm_h3_clean.select("cellid", "max_chm_z"), "max_chm_z")
chm_filled_land = chm_filled.join(land_cells, "cellid", "inner")
filled_count = chm_filled_land.count()
post_fill_max = chm_filled_land.selectExpr("max(max_chm_z)").first()[0]
post_fill_min = chm_filled_land.selectExpr("min(max_chm_z)").first()[0]
post_fill_max = post_fill_max if post_fill_max is not None else float("nan")
post_fill_min = post_fill_min if post_fill_min is not None else float("nan")
print(f"CHM cells: {clean_cells} present → {chm_filled.count()} filled "
      f"→ {filled_count} after land mask")
print(f"Post-fill max_chm_z range: [{post_fill_min:.1f}, {post_fill_max:.1f}] m "
      f"(ceiling {MAX_CHM_M} m)")

In [0]:
if RENDER_H3:
    filled_gdf = cells_as_gdf(
        chm_filled_land.withColumn("res", F.lit(H3_RESOLUTION)),
        "cellid", extra_cols=["max_chm_z"],
        dissolve_engine="product",
    )
    plot_static(
        filled_gdf, column="max_chm_z", cmap="viridis",
        title=f"CHM — max canopy height with gaps filled (h3_cellfill k=1 IDW, res {H3_RESOLUTION})",
    )


## Part 3 — T5: Composite and comparison renders

Ties the pipeline together with three multi-layer renders:

1. **CHM raster + H3 overlay** — a 1 m-pixel canopy surface tile composited with
   the H3-gridded max canopy heights, showing how the hexagonal aggregation
   approximates the raw pixel surface at coarser resolution.
2. **DEM vs DSM side-by-side** — bare-earth vs surface elevation bands in one
   figure to highlight where buildings and canopy push DSM above DEM.
3. **LiDAR point cloud narrative** — the raw RGB-coloured input that feeds
   nb1/nb2, connecting the source point cloud to the H3-gridded outputs above.

In [0]:
if RENDER_H3:
    # --- T5 Cell 1: CHM raster tile + CHM H3 cells composite ---
    # Overlay the 1 m-pixel CHM surface tile (raster_layer) with the H3-gridded max
    # canopy heights (vector_layer) for Golden Gate Park — the same AOI used throughout.
    _gx2, _gy2 = lonlat_to_3857(-122.486, 37.769)
    _tx2, _ty2  = int((_gx2 - X0) / TILE_M), int((_gy2 - Y0) / TILE_M)
    chm_tile = chm.where((F.col("tx") == _tx2) & (F.col("ty") == _ty2)).first()
    chm_cells_gdf = cells_as_gdf(
        chm_h3, "cellid", extra_cols=["max_chm_z"], dissolve_engine="product"
    )
    if chm_tile:
        plot_static(
            [
                raster_layer(chm_tile["chm"], band=1, cmap="Greens", opacity=0.6,
                             label="CHM (1 m pixel)"),
                vector_layer(chm_cells_gdf, geom_col="geometry", column="max_chm_z",
                             cmap="viridis", opacity=0.5,
                             label=f"H3 cells (res {H3_RESOLUTION}, max canopy ht)"),
            ],
            title="CHM: 1 m pixel surface vs H3-gridded max canopy height",
        )
    else:
        print(f"No CHM tile at tx={_tx2}, ty={_ty2} in DEMO window — skipped.")


In [0]:
if RENDER_H3:
    # --- T5 Cell 2: DEM vs DSM H3 cells side-by-side ---
    # Elevation bands from the bare-earth DTM (DEM) and the surface model (DSM,
    # which includes buildings and canopy) in one figure.  Cells where DSM exceeds DEM
    # reveal structure and vegetation height above bare ground.
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    dem_gdf = cells_as_gdf(
        dem_h3, "cellid", extra_cols=["band_level"],
        dissolve_by="band_level", dissolve_engine="product",
    )
    dsm_gdf = cells_as_gdf(
        dsm_h3, "cellid", extra_cols=["band_level"],
        dissolve_by="band_level", dissolve_engine="product",
    )
    plot_static(dem_gdf, column="band_level", cmap="terrain",
                title="DEM — bare earth elevation bands", ax=axes[0])
    plot_static(dsm_gdf, column="band_level", cmap="terrain",
                title="DSM — surface (buildings + canopy) elevation bands",
                ax=axes[1], basemap=False)
    plt.tight_layout()


## Part 3 — T9: Full-SF scale-up, profiling, and library-fix flags

Runs the complete San Francisco AOI (`DEMO=False`, all tiles at H3 res 9). Each H3-gridding path is timed independently. If a path exceeds the 30-minute budget, a structured `FLAG` line is emitted and the cell raises — the flag is a tracked library-fix obligation, not a reason to reduce the AOI.

Expected timings on Serverless (env 6):
- DEM/DSM isoband path: 5–20 min (all tiles × isoband + h3_try_coverash3)
- CHM rastertogridmax: 5–15 min (all tiles × LATERAL UDTF)
- cellfill: < 2 min (driver-side aggregation over ~1 150 cells)

In [0]:
# ── FULL-SF SCALE-UP — T9 rebuilds tables at full SF scale ───────────────────────────
# T9 sets DEMO=False and FORCE_RELOAD=True, then calls _persist() for each surface —
# which runs saveAsTable.  Reported timings INCLUDE the Delta write (not compute-only).
# T2/T3 above are the demo-scale build; T9 is the full-SF overwrite.
# A double full-SF build only occurs when DEMO=False is injected globally for the whole
# notebook run (intentional — used for profiling; understood as a 2× cost).
# Expected timings at H3 res 9 over all of SF on Serverless:
#   DEM/DSM isoband path:  5–20 min (336 tiles × isoband + h3_try_coverash3)
#   CHM rastertogridmax:   5–15 min (336 tiles × LATERAL UDTF)
#   cellfill:              < 2 min  (driver-side aggregation over ~1,150 cells)
# If a path exceeds 30 min or errors, that is a required library fix — see flags below.
DEMO = False
FORCE_RELOAD = True    # T9 profiles compute via _persist() write — uses repartition fix
# Re-read source tables under DEMO=False so _tbl() resolves full (non-demo) table names.
dtm_in = spark.table(_tbl(SRC_PREFIX, "dtm"))
dsm_in = spark.table(_tbl(SRC_PREFIX, "dsm"))
chm_in = spark.table(_tbl(SRC_PREFIX, "chm"))  # full tile sets
print(f"Full-SF: {dtm_in.count()} DTM | {dsm_in.count()} DSM | {chm_in.count()} CHM tiles")

In [0]:
import time
t0 = time.time()
dem_cells_full = _surface_to_h3_cells(dtm_in, "dtm", breaks_arr, H3_RESOLUTION, land_wkb)
dem_h3_full    = _persist(dem_cells_full, "dem", res=H3_RESOLUTION)   # → wc_h3_dem_res10; repartition fix prevents OOM
dem_elapsed    = time.time() - t0
dem_count      = dem_h3_full.count()
print(f"DEM full-SF: {dem_count} H3 cells in {dem_elapsed:.1f}s "
      f"({dem_count / dem_elapsed:.0f} cells/s)")
assert dem_count > 500, f"Expected >500 DEM H3 cells at res 9 over full SF, got {dem_count}"

In [0]:
t0 = time.time()
dsm_cells_full = _surface_to_h3_cells(dsm_in, "dsm", breaks_arr, H3_RESOLUTION, land_wkb)
dsm_h3_full    = _persist(dsm_cells_full, "dsm", res=H3_RESOLUTION)   # → wc_h3_dsm_res10; repartition fix prevents OOM
dsm_elapsed    = time.time() - t0
dsm_count      = dsm_h3_full.count()
print(f"DSM full-SF: {dsm_count} H3 cells in {dsm_elapsed:.1f}s")
assert dsm_count > 500, f"Expected >500 DSM H3 cells, got {dsm_count}"

In [0]:
t0 = time.time()
# repartition(64, 'tx') here matches the DEMO cell; 512 is handled inside
# _surface_to_h3_cells for DEM/DSM; CHM uses a direct SQL path so we keep
# the explicit repartition here.
chm_4326_full = chm_in.select(
    "tx", "ty", rx.rst_transform("chm", F.lit(4326)).alias("chm_4326")
).repartition(512, "tx", "ty")
chm_4326_full.createOrReplaceTempView("chm_4326_tiles_full")
# ANSI LATERAL form (verified schema: band INT, cellID LONG, measure DOUBLE).
chm_cells_full = spark.sql(f"""
    SELECT t.cellID AS cellid, t.measure AS chm_z
    FROM   chm_4326_tiles_full,
           LATERAL gbx_rst_h3_rastertogridmax(chm_4326, {H3_RESOLUTION}) t
    WHERE  t.band = 1
""").groupBy("cellid").agg(F.max("chm_z").alias("max_chm_z")) \
   .withColumn("res", F.lit(H3_RESOLUTION))
chm_h3_full   = _persist(chm_cells_full, "chm", res=H3_RESOLUTION)   # → wc_h3_chm_res10; repartition prevents OOM
chm_elapsed   = time.time() - t0
chm_count     = chm_h3_full.count()
print(f"CHM full-SF: {chm_count} H3 cells in {chm_elapsed:.1f}s "
      f"({chm_count / chm_elapsed:.0f} cells/s)")
assert chm_count > 500, f"Expected >500 CHM H3 cells, got {chm_count}"

In [0]:
t0 = time.time()
# MAX_CHM_M clamp applies at full scale (same ceiling as DEMO path, cell T4).
chm_h3_full_clean = chm_h3_full.where(F.col("max_chm_z") <= MAX_CHM_M)
chm_filled_full = _cellfill_depth(chm_h3_full_clean.select("cellid", "max_chm_z"), "max_chm_z")
chm_filled_land_full = chm_filled_full.join(land_cells, "cellid", "inner")
fill_elapsed = time.time() - t0
print(f"CHM cellfill full-SF: {chm_h3_full_clean.count()} → {chm_filled_land_full.count()} cells "
      f"in {fill_elapsed:.1f}s")

In [0]:
print("| Path | Tiles | Output cells | Wall-clock (s) |")
print("|------|-------|-------------|----------------|")
print(f"| DEM isoband→H3  | {dtm_in.count()} | {dem_count}  | {dem_elapsed:.0f} |")
print(f"| DSM isoband→H3  | {dsm_in.count()} | {dsm_count}  | {dsm_elapsed:.0f} |")
print(f"| CHM rastertogridmax | {chm_in.count()} | {chm_count} | {chm_elapsed:.0f} |")
print(f"| CHM cellfill    | —     | {chm_filled_land_full.count()} | {fill_elapsed:.0f} |")

In [0]:
WALL_CLOCK_LIMIT_S = 1800   # 30 min per path = upper bound before flagging
flags = []
if dem_elapsed > WALL_CLOCK_LIMIT_S:
    flags.append("FLAG: rst_isoband / h3_try_coverash3 chain exceeds 30 min on full-SF "
                 f"({dem_elapsed:.0f}s). Requires gbx library perf fix: "
                 "batch rst_isoband + distributed H3 coverage, not driver-side loop.")
if chm_elapsed > WALL_CLOCK_LIMIT_S:
    flags.append("FLAG: gbx_rst_h3_rastertogridmax LATERAL UDTF exceeds 30 min on full-SF "
                 f"({chm_elapsed:.0f}s). Requires gbx library perf fix: "
                 "UDTF vectorization or Arrow batch output.")
if fill_elapsed > 120:
    flags.append("FLAG: h3_cellfill exceeds 2 min on full-SF "
                 f"({fill_elapsed:.0f}s). Requires gbx library perf fix: "
                 "driver-side aggregation may not scale; explore distributed implementation.")
if flags:
    for f in flags:
        print(f)
    raise RuntimeError(
        f"{len(flags)} scale flag(s) raised — see above. "
        "Address as library fixes before shipping this example."
    )
else:
    print("Full-SF scale-up PASSED — all paths within time budget.")

In [0]:
if RENDER_H3:
    dem_gdf_full = cells_as_gdf(
        dem_h3_full, "cellid", extra_cols=["band_level"],
        dissolve_by="band_level", dissolve_engine="product",
    )
    plot_static(dem_gdf_full, column="band_level", cmap="terrain",
                title=f"Full SF — DEM elevation bands, H3 res {H3_RESOLUTION}")
    
    chm_gdf_full = cells_as_gdf(
        chm_h3_full, "cellid", extra_cols=["max_chm_z"], dissolve_engine="product"
    )
    plot_static(chm_gdf_full, column="max_chm_z", cmap="viridis",
                title=f"Full SF — CHM max canopy height, H3 res {H3_RESOLUTION}")
